In [ ]:
##   start from inference

In [1]:
import pathlib
import joblib
import pandas as pd
import numpy as np
from datetime import datetime
import yaml


import rest_api.schemas
import features.util_feature
import features.feature_engineer

E:\Learning_course\MLOps\coursera_packt\greenHouse_Emission_predictor\src\domain\usstates.json
True

E:\Learning_course\MLOps\coursera_packt\greenHouse_Emission_predictor\src\domain\source_types.json



In [ ]:
request_data = {
  "capacity": 250.0,
  "capacity_factor": 0.65,
  "activity": 120000.0,
  "source_type": "coal",
  "state": "Georgia",
  "area": 153910,
  "pop2020": 10711908
}

pRequest_df = pd.DataFrame( [request_data] )
# pRequest_df = pd.DataFrame( [request_data.dict()] )
pRequest_df

In [ ]:
# BASE_DIR = pathlib.Path(__file__).resolve().parents[2]
BASE_DIR = pathlib.Path( r'E:\Learning_course\MLOps\coursera_packt\greenHouse_Emission_predictor' )
MODEL_DIR = BASE_DIR / 'models' / 'trained'

MODEL_PATH = MODEL_DIR / 'greenhouse_emission_predict_model.pkl'
PREPROCESSOR_PATH = MODEL_DIR / 'preprocessor.pkl'
MODEL_CONFIG_PATH = BASE_DIR / 'configs' / 'model_config.yaml'

In [ ]:
model = joblib.load(MODEL_PATH)
preprocessor = joblib.load(PREPROCESSOR_PATH)

In [ ]:

with open(MODEL_CONFIG_PATH, 'r', encoding='utf-8') as f:
    model_cfg = yaml.safe_load(f)

In [ ]:
### feature engg


# Apply EXACT SAME feature engineering as training
featured_df = features.util_feature.create_features(pRequest_df)
print( featured_df.shape )

# Apply saved preprocessor (OHE with fixed category space)
# xT = preprocessor.transform(featured_df)

# Build the same engineered feature table as in feature_engineer.py
engineered_x_df, engineered_cols = features.util_feature.transform_to_engineered_df(
    preprocessor=preprocessor,
    xx= featured_df,
    remaining_features_ls= features.feature_engineer.REMAINING_Features_ls 
    )

engineered_x_df

In [ ]:
yhat = int( model.predict(engineered_x_df.values)[0] )

confidence_interval = [ round( yhat * 0.9, 0 ),  round( yhat * 1.1, 0) ]
confidence_interval


#### fastapi

In [2]:
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware as fapi_CORSMiddleware

import rest_api.inference
import rest_api.schemas


In [6]:
### initialize FastAPI app

fapi_app = FastAPI(

    title= 'Green House Emission Prediction API',
    description= 'An API that serves ML model for predicting green house emission quantity based on power plant features.',
    version= '1.0.0',
    contact= {
        'name': 'Madhur Dev',
        'website': 'https://www.madhurdev.com',
        'linkedin': 'www.linkedin.com/in/madhurdev',
        'dashboard-portfolio':  'https://madhurdev.com/dashboards',
        'github': 'https://github.com/madhurdevkota'
    },
    license_info= {
        'name': 'Apache 2.0',
        'url': 'https://www.apache.org/licenses/LICENSE-2.0.html'
    }
)


type(fapi)

fastapi.applications.FastAPI

In [7]:
### add cors middleware
fapi_app.add_middleware(
    fapi_CORSMiddleware,
    allow_origins= ['*'],
    allow_credentials= True,
    allow_methods= ['*'],
    allow_headers= ['*'],
)

In [11]:
## health check endpoint
@fapi_app.get( '/health', response_model = dict )
async def health_check_endpoint():
    return { 'status': 'ok', 'model_loaded': True }


## predict endpoint
@fapi_app.post( '/predict', response_model= rest_api.schemas.Emission_Prediction_Response )
async def predict_endpoint( requests: rest_api.schemas.Emission_Prediction_Request ):
    return rest_api.inference.predict_emission( request= requests )

## batch predict endpoint
@fapi_app.post( '/predict-batch', response_model= list )
async def batchPredict_endpoint( requests: list[ rest_api.schemas.Emission_Prediction_Request ] ):
    return rest_api.inference.batch_predict_emission( requests= requests )

In [13]:
import streamlit as st
import requests
import time
import os
import socket

In [15]:
df = pd.read_csv( r'E:\Learning_course\MLOps\coursera_packt\greenHouse_Emission_predictor\data\processed\processed_data.csv' )

In [20]:
des_df = df.describe()

In [ ]:
des_df.index
##

Index(['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max'], dtype='object')

In [26]:
### import matplotlib with inline
%matplotlib inline
import matplotlib.pyplot as plt

In [29]:
## choose index which have these values [  'min', 'max' ]
selected_stats = [ 'min', 'max' ]
stat_options = des_df.index.tolist()
## filter stat_options to only have selected_stats
stat_options = [ stat for stat in stat_options if stat in selected_stats ]
stat_options
##
des_df.loc[ stat_options ]

# df['area'].plot( kind='hist', bins=50, title='State Area Distribution' )
# plt.show()


,emissions_quantity,emissions_factor,capacity,capacity_factor,activity,area,pop2020
min,2000.0,0.308,2.0,0.114,2000.0,0.018377,576851.0
max,14673000.0,1.487,4330.0,0.694,15727000.0,65.363350,39538223.0
